# Run Match

In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [3]:
from timeutils import Timestat
from master import MasterParams, MasterDBs, MasterPaths, MasterMetas
from musicdb import MusicDBIO
from gate import MusicDBGate
from pandas import DataFrame, Series, concat
import musicdb
from ioutils import HTMLIO, FileIO
from listUtils import getFlatList
import swifter
import dask.dataframe as dd
from match import getLevenshtein, MatchDB, MatchDBDataIO, AlbumReq, MatchID
mp     = MasterParams(verbose=True)
io     = FileIO()
pdbio  = MusicDBIO()
gate   = MusicDBGate()
mdbios = gate.getIO(None)
mmeDF  = pdbio.getData()

MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM']
MasterParams()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb


In [4]:
baseDB    = "RateYourMusic"
baseDB    = "LastFM"
baseReq   = {baseDB: AlbumReq(top=500)}

compareDBs  = ["RateYourMusic", "LastFM", "Spotify", "Genius", "Discogs", "MusicBrainz", "Deezer", "MetalArchives"]
compareReqs = {compareDB: AlbumReq(min=3) for compareDB in compareDBs if compareDB not in [baseDB]}
compareDBs  = list(compareReqs.keys())

albumReqs = {**baseReq, **compareReqs}
mediaTypes = ["Album", "SingleEP"]
mediaTypes = ["{0}Media".format(mediaType) for mediaType in mediaTypes]
reqs       = {"Media": mediaTypes, "Albums": albumReqs, "Mask": baseDB, "NPart": 3, "Match": {"Artist": 0.8, "Medium": 2, "Tight": 1}}

print("Run Params:")
print("  ==> DBs:   [{0}] x {1}]".format(baseDB,list(compareReqs.keys())))
print("  ==> Media: {0}".format(mediaTypes))
print("  ==> Match: {0}".format(reqs["Match"]))

Run Params:
  ==> DBs:   [LastFM] x ['RateYourMusic', 'Spotify', 'Genius', 'Discogs', 'MusicBrainz', 'Deezer', 'MetalArchives']]
  ==> Media: ['AlbumMedia', 'SingleEPMedia']
  ==> Match: {'Artist': 0.8, 'Medium': 2, 'Tight': 1}


In [5]:
mdb = MatchDB(baseDB=baseDB, compareDBs=compareDBs, reqs=reqs)
mdb.match()

###############################################################################################################################################################################
######################################################################  MatchDB()  ############################################################################################
###############################################################################################################################################################################
Process [Matching [LastFM] Against ['RateYourMusic', 'Spotify', 'Genius', 'Discogs', 'MusicBrainz', 'Deezer', 'MetalArchives']] Start    ==> Time Is 2022-04-05 16:07:25

------------------------------------------------------------------------------------------------------------------------------------------------------
---------------------------------------------------------------------- LastFM ------------------------------------------------------------------------


      ==>  285290 Artists To Match <==
      ==>    1013 Max Albums
      ==>       3 Min Albums
Process [String Matching 491 [LastFM] x 285290 [Discogs] Artist Names] Start    ==> Time Is 2022-04-05 17:02:45
   Found 491 Name Results
   Found 421 Artists With One Or More Matches
   Found 1586 Possible Matches
Process [String Matching 491 [LastFM] x 285290 [Discogs] Artist Names] Ran For 2 Minutes and 58 Seconds    ==> Time Is 2022-04-05 17:05:43
Process [String Matching 421 [LastFM] Album Names] Start    ==> Time Is 2022-04-05 17:05:43
  ==> Found    9 Multiple Good Matches
  ==> Found  291 Good Matches
  ==> Found   36 Near Matches
Process [String Matching 421 [LastFM] Album Names] Ran For 31 Seconds    ==> Time Is 2022-04-05 17:06:14
Found  291 [Discogs] Artists To Match Against LastFM
  MatchDBData(Discogs).getData(ids=291):
      ==>     291 Artists To Match <==
      ==>     302 Max Albums
      ==>       3 Min Albums
  MatchDBData(LastFM).getData(AlbumReq)
      ==>  115963 Arti

   Found 8 Name Results
   Found 8 Artists With One Or More Matches
   Found 17 Possible Matches
Process [String Matching 8 [MetalArchives] x 115963 [LastFM] Artist Names] Ran For 56 Seconds    ==> Time Is 2022-04-05 17:21:20
Process [String Matching 8 [MetalArchives] Album Names] Start    ==> Time Is 2022-04-05 17:21:20
  ==> Found    0 Multiple Good Matches
  ==> Found    8 Good Matches
  ==> Found    0 Near Matches
Process [String Matching 8 [MetalArchives] Album Names] Ran For 1 Second    ==> Time Is 2022-04-05 17:21:21
Process [Matching [LastFM] Against ['RateYourMusic', 'Spotify', 'Genius', 'Discogs', 'MusicBrainz', 'Deezer', 'MetalArchives']] Has Run For 1 Hour and 13 Minutes.
Process [Matching [LastFM] Against ['RateYourMusic', 'Spotify', 'Genius', 'Discogs', 'MusicBrainz', 'Deezer', 'MetalArchives']] Ran For 1 Hour and 13 Minutes    ==> Time Is 2022-04-05 17:21:21


# Get Master IDs

In [11]:
mid = MatchID(baseDB, mdb, pdbio, verbose=True)
mid.join()
mid.getMasterID()
mid.joinMaster()

MatchID()
  ==> Found 405 x 7 Matched Entries/DBs
  ==> Getting Master ID Lookup
  ==> Mapping Master ID Lookup From Matchd DB IDs
  ==> Merging 320/405 Entries With PanDB
  ==> Adding 17/405 New Good Entries To PanDB
  ==> Adding 0/405 New OK Entries To PanDB
  ==> Adding 269/405 New Loose Entries To PanDB


In [12]:
mid.mergeMaster()

  ==> Set [iiiiiiiiXXX0004472XXX1/LastFM] to [10581639807665]
  ==> Set [iiiiiiiiXXX0004472XXX1/Discogs] to [94769]
  ==> Set [iiiiiiiiXXX0004472XXX1/MusicBrainz] to [240934054690878471155118420210241183409]
  ==> Set [hhhhhhhhXXX0014040XXX1/LastFM] to [10732580207939]
  ==> Set [hhhhhhhhXXX0014040XXX1/Discogs] to [959750]
  ==> Set [llllllllXXX0009260XXX1/LastFM] to [10849428454699]
  ==> Set [llllllllXXX0009260XXX1/Discogs] to [6723982]
  ==> Set [llllllllXXX0009260XXX1/Deezer] to [1275336]
  ==> Set [jjjjjjjjXXX0001215XXX1/LastFM] to [10905409546102]
  ==> Set [jjjjjjjjXXX0001215XXX1/Discogs] to [178942]
  ==> Set [aaaaaaaaXXX0016235XXX1/LastFM] to [10910142426311]
  ==> Set [aaaaaaaaXXX0016235XXX1/RateYourMusic] to [1575852]
  ==> Set [aaaaaaaaXXX0016235XXX1/Spotify] to [32ZDarYu9wVcwLNdVX4IHP]
  ==> Set [aaaaaaaaXXX0016235XXX1/Discogs] to [5224524]
  ==> Set [aaaaaaaaXXX0016235XXX1/Deezer] to [4518550]
  ==> Set [00029579XXX0013163XXX1/LastFM] to [11493479037796]
  ==> Set [000295

  ==> Set [zzzzzzzzXXX0000858XXX1/Genius] to [1403962]
  ==> Set [zzzzzzzzXXX0000858XXX1/Discogs] to [6158394]
  ==> Set [zzzzzzzzXXX0000858XXX1/Deezer] to [11055882]
  ==> Set [kkkkkkkkXXX0004842XXX1/LastFM] to [90462896698870]
  ==> Set [kkkkkkkkXXX0004842XXX1/Discogs] to [1269999]
  ==> Set [kkkkkkkkXXX0004842XXX1/Deezer] to [3009581]
  ==> Set [rrrrrrrrXXX0013728XXX1/LastFM] to [90905188298338]
  ==> Set [rrrrrrrrXXX0013728XXX1/Discogs] to [3044533]
  ==> Set [rrrrrrrrXXX0010397XXX1/LastFM] to [91183758960638]
  ==> Set [rrrrrrrrXXX0010397XXX1/Spotify] to [6U1dV7aL68N7Gb0Naq34V5]
  ==> Set [rrrrrrrrXXX0010397XXX1/Genius] to [1917162]
  ==> Set [rrrrrrrrXXX0010397XXX1/Deezer] to [67515242]
  ==> Set [hhhhhhhhXXX0011064XXX1/LastFM] to [9169444218802]
  ==> Set [hhhhhhhhXXX0011064XXX1/RateYourMusic] to [1084548]
  ==> Set [hhhhhhhhXXX0011064XXX1/Spotify] to [6FfZaHz07OsknWNdtdan5R]
  ==> Set [hhhhhhhhXXX0011064XXX1/Genius] to [1162801]
  ==> Set [hhhhhhhhXXX0011064XXX1/Discogs] to [46

  ==> Added New Row [ArtistName           Carlos Martinez
LastFM                28399309453543
Spotify       1ZMERX5E7xoX502UNoy6ux
Deezer                     104755642
Name: acffd11e-5cd3-47e4-aa2f-72d2c5a073c0, dtype: object]
  ==> Added New Row [ArtistName    Sachet Tandon
LastFM        2842636894627
Genius              1833277
Deezer             14114495
Name: a2e46ab6-177a-4404-9e97-a89b5f379a66, dtype: object]
  ==> Added New Row [ArtistName        Rick Rubin
LastFM        29234524050637
Genius                 27794
Deezer               4416661
Name: 6782ada4-fe8a-4a05-b19d-f5f1c1493c8b, dtype: object]
  ==> Added New Row [ArtistName       musikFabrik
LastFM        35577248349570
Discogs               462177
Deezer                604611
Name: 97c03460-75f1-4000-b724-7afb7dbc1cad, dtype: object]
  ==> Added New Row [ArtistName      Michael Berg
LastFM        36051259465801
Genius                607163
Deezer               4474421
Name: 6754de1f-9bc0-4cab-bc36-726b7ea8c1c4, dtype: 